In [2]:
# %pip install earthengine-api
# !pip install --upgrade earthengine-api
import ee

# Authenticate and initialize the Earth Engine API
ee.Authenticate(scopes=[ "https://www.googleapis.com/auth/earthengine", "https://www.googleapis.com/auth/devstorage.full_control" ])
ee.Initialize(project='your-gcp-project-id')


In [3]:
import json
import glob

# Load all DHA Phase 1-9 GeoJSONs from the 'societies/' directory
dha_phase_geometries = []
for i in range(1, 10):
    geojson_path = f'societies/dha_phase{i}.geojson'
    with open(geojson_path) as f:
        geojson = json.load(f)
        # Assuming each file contains a single feature
        geom = ee.Geometry(geojson['features'][0]['geometry'])
        dha_phase_geometries.append(geom)

# Merge all DHA phases into a single geometry
dha_societies = ee.Geometry.MultiPolygon([geom.coordinates().getInfo() for geom in dha_phase_geometries])

# Get NDVI (using Sentinel-2)
sentinel2 = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterDate('2023-01-01', '2023-12-31') \
    .filterBounds(dha_societies) \
    .median()

ndvi = sentinel2.normalizedDifference(['B8', 'B4']).rename('NDVI')

# Get WorldPop population density (latest available)
pop_density = ee.ImageCollection('WorldPop/GP/100m/pop') \
    .filterDate('2020-01-01', '2020-12-31') \
    .first() \
    .select('population')

# Get road network (OpenStreetMap)
roads = ee.FeatureCollection('projects/ee-geo-demo/assets/OSM_roads_pakistan') \
    .filterBounds(dha_societies)

# Export NDVI, population density, and road network for DHA Phase 1-9
task_ndvi = ee.batch.Export.image.toDrive(
    image=ndvi.clip(dha_societies),
    description='DHA_Phase1_9_NDVI',
    folder='EarthEngineExports',
    fileNamePrefix='dha_phase1_9_ndvi',
    region=dha_societies,
    scale=10,
    maxPixels=1e13
)
task_ndvi.start()

task_pop = ee.batch.Export.image.toDrive(
    image=pop_density.clip(dha_societies),
    description='DHA_Phase1_9_PopDensity',
    folder='EarthEngineExports',
    fileNamePrefix='dha_phase1_9_popdensity',
    region=dha_societies,
    scale=100,
    maxPixels=1e13
)
task_pop.start()

task_roads = ee.batch.Export.table.toDrive(
    collection=roads,
    description='DHA_Phase1_9_Roads',
    folder='EarthEngineExports',
    fileNamePrefix='dha_phase1_9_roads'
)
task_roads.start()


EEException: Project 'projects/your-gcp-project-id' not found or deleted.